<img src="https://raw.githubusercontent.com/pasdptt/public_teaching_tools4DS/main/assets/brand/logo-160.png" width="80" align="right" alt="Course logo">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pasdptt/public_teaching_tools4DS/blob/main/modules/m07-networks/lab/lab7a_graphs_paths.ipynb)

# Lab 7A — Graphs from tables, paths and Dijkstra

In [ ]:
# Setup: run this cell first (Google Colab or a local Jupyter kernel).
COURSE_REF = "main"  # version of the course repository to install the helper package from
try:
    import tsds
except ImportError:  # first run in a fresh Colab runtime: install the course helper package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"tsds @ git+https://github.com/pasdptt/public_teaching_tools4DS@{COURSE_REF}#subdirectory=src"], check=True)
    import tsds

tsds.env.check()          # prints versions; warns if a package is outside the tested range
SEED = tsds.seed_everything()
rng = tsds.rng()

import heapq
import math
import numpy as np
import pandas as pd
import networkx as nx

**Course:** 040563107 Tools for Statistical Data Science · **Module m07** (week 1 of 2) · **Time:** 3 hours · **Practice lab** (not graded; use the self-checks)

**After this lab you can:**
* build a **directed, weighted** graph from tabular records, deciding what nodes, edges and weights mean (objective m07-O1);
* compute in- and out-degrees and check the handshake identity for directed graphs (O2);
* show what goes wrong when directed, repeated records are collapsed into an undirected simple graph (O1, failure analysis);
* implement Dijkstra's algorithm and check it against networkx (O3);
* convert clock times safely, and spot a silent parsing change between pandas versions (failure analysis).

**Two steps:** in the lab session we **run through** this lab together. Then, as homework, you **do it again on your own** from a fresh
copy and upload the `.ipynb` to the Lab 7A practice assignment in Microsoft Teams.

**How to work:**
1. Run the setup cell first.
2. Complete every cell marked **Task**. Replace `raise NotImplementedError()` with your code, then run the **Self-check** cell below it; it prints `OK` when your answer passes.
3. Write the answers marked **Q** after *Your answer:* (at most 100 words each).
4. Before you finish, use *Runtime → Restart session and run all*. The notebook must run top to bottom without errors (the first data load takes about 30 s).
5. Fill in the **AI-use log** at the end.

## Part 1 — From a table of flights to a directed, weighted graph (about 40 minutes)

The data are all 336,776 flights that departed New York City's three airports (EWR, JFK, LGA) in 2013. One **row** is one
flight. A graph needs decisions:

* **nodes:** airports;
* **edges:** a **directed** edge $u \to v$ when at least one flight flew from $u$ to $v$ (direction matters: a flight JFK → LAX
  is not a flight LAX → JFK);
* **weights:** many rows share an edge, so they must be **aggregated**: here the number of flights and the mean air time.

Cancelled flights have no `air_time`; we keep only flights that flew.

In [ ]:
flights = tsds.data.load("nycflights13_flights")
flown = flights.dropna(subset=["air_time"])
print(f"{len(flights):,} rows; {len(flown):,} flown; origins: {sorted(flights.origin.unique())}")
flown[["origin", "dest", "carrier", "air_time", "distance"]].head()

In [ ]:
# Task T1
# (a) routes: one row per (origin, dest) pair in `flown`, with columns origin, dest,
#     n_flights (number of flights), mean_air_time (mean of air_time, minutes) and distance (miles; the same on every row of a pair)
# (b) G: an nx.DiGraph built from routes, with edge attributes n_flights, mean_air_time, distance
#     (nx.from_pandas_edgelist(..., create_using=nx.DiGraph))
# YOUR CODE HERE
raise NotImplementedError()
print(f"{G.number_of_nodes()} airports, {G.number_of_edges()} directed routes")
print("out-degree of the NYC airports:", dict(G.out_degree(["EWR", "JFK", "LGA"])))
in_deg = pd.Series(dict(G.in_degree())).drop(["EWR", "JFK", "LGA"])
print("in-degree of the destinations (how many NYC airports serve them):", in_deg.value_counts().sort_index().to_dict())

In [ ]:
# Self-check for task T1: run it; it should print OK.
assert isinstance(G, nx.DiGraph) and G.number_of_nodes() == 107 and G.number_of_edges() == 223
assert G.edges["JFK", "LAX"]["n_flights"] == routes.query("origin == 'JFK' and dest == 'LAX'").n_flights.item()
assert sum(d for _, d in G.in_degree()) == sum(d for _, d in G.out_degree()) == G.number_of_edges()
assert routes.n_flights.sum() == len(flown)
print("OK")

**The handshake identity for directed graphs.** Every directed edge adds exactly 1 to the out-degree of its tail and 1 to the
in-degree of its head, so $\sum_v d^{+}(v) = \sum_v d^{-}(v) = |E|$. (This direct count is the proof; the course's old notes
argued by contradiction, audit M20.) The self-check above verified it on the flights graph; the task cell also printed the
in-degrees of the destinations.

**Q1.** **Interpretation.** Describe the shape of this network using the out-degrees and the in-degree counts. What questions about air
travel can it answer, and which can it not, given how the data were collected?

*Your answer:* 

## Part 2 — Failure analysis: collapsing directed records into an undirected graph (about 35 minutes)

The course's old graph notebook built its network with one line (audit issue M19). Run it on all rows.

In [ ]:
# ---- the old notebook's construction (do not edit) ----
FG = nx.from_pandas_edgelist(flights, source="origin", target="dest", edge_attr=True)
print(type(FG).__name__, FG.number_of_nodes(), "nodes,", FG.number_of_edges(), "edges")
print("FG says the quickest route JAX -> DFW is", nx.dijkstra_path(FG, "JAX", "DFW", weight="air_time"))

`nx.Graph` is **undirected** and **simple**: it keeps one edge per pair of airports, and each new row **overwrites** the edge's
attributes, so the edge keeps the attributes of whichever row came last.

In [ ]:
# Task T2
# (a) fg_air_time: the air_time stored on FG's JFK-LAX edge; g_air_time: the mean_air_time on G's JFK -> LAX edge
# (b) reach_directed: the number of ordered pairs (u, v), u != v, with a directed path from u to v in G
#     reach_undirected: the same count in FG (any path)
#     (hint: nx.descendants(G, u) and nx.node_connected_component(FG, u))
# (c) jax_dfw_directed: True or False, is there a directed path JAX -> DFW in G?
# YOUR CODE HERE
raise NotImplementedError()
print(f"JFK-LAX air time: FG {fg_air_time} (one row) vs G {g_air_time:.1f} (mean of {G.edges['JFK', 'LAX']['n_flights']:,} flights)")
print(f"reachable ordered pairs: directed {reach_directed:,} | undirected {reach_undirected:,}; JAX -> DFW possible? {jax_dfw_directed}")

In [ ]:
# Self-check for task T2: run it; it should print OK.
assert fg_air_time == flights.query("origin == 'JFK' and dest == 'LAX'").air_time.iloc[-1], "FG keeps the LAST row's attributes"
assert abs(g_air_time - 329.2) < 0.1
assert reach_directed == 223 and reach_undirected == 107 * 106
assert not jax_dfw_directed, "no flight leaves JAX in these data"
print("OK")

## Part 3 — Dijkstra's algorithm (about 45 minutes)

For non-negative weights, Dijkstra's algorithm finds the shortest distance from a source to every node:

1. $d(s) = 0$ and $d(v) = \infty$ for every other node; put $(0, s)$ in a priority queue.
2. Repeatedly remove the node $u$ with the **smallest** tentative distance. If it was already finalised, skip it; otherwise
   finalise it.
3. For each edge $u \to v$ with weight $w$: if $d(u) + w < d(v)$, set $d(v) = d(u) + w$ and add $(d(v), v)$ to the queue.

**Why it is correct:** when $u$ is removed with the smallest tentative distance, any other path to $u$ must leave the finalised
set through some node whose tentative distance is already at least $d(u)$, and non-negative weights can only add to that. So
$d(u)$ is final. With a negative weight this argument fails. Use `math.inf` for "infinity", not a large number such as 32767
(the old notes did, audit M20).

The graph below is the one we trace by hand in the lecture. `adj[u]` lists `(v, weight)` pairs.

In [ ]:
adj_small = {
    "A": [("B", 4), ("C", 2)],
    "B": [("D", 5)],
    "C": [("B", 1), ("D", 8), ("E", 10)],
    "D": [("E", 2), ("F", 6)],
    "E": [("F", 2)],
    "F": [],
}

In [ ]:
# Task T3
# dijkstra(adj, source): adj is a dict node -> list of (neighbour, weight >= 0). Return a dict node -> shortest distance from
# source, with math.inf for nodes that cannot be reached. Use heapq as the priority queue.
def dijkstra(adj, source):
    # YOUR CODE HERE
    raise NotImplementedError()


print(dijkstra(adj_small, "A"))

In [ ]:
# Self-check for task T3: run it; it should print OK.
assert dijkstra(adj_small, "A") == {"A": 0, "B": 3, "C": 2, "D": 8, "E": 10, "F": 12}
assert dijkstra(adj_small, "D")["A"] == math.inf, "A cannot be reached from D"
for trial in range(5):
    R = nx.gnp_random_graph(40, 0.08, seed=trial, directed=True)
    w = np.random.default_rng(trial).uniform(0, 10, R.number_of_edges())
    for (u, v), wt in zip(R.edges(), w):
        R.edges[u, v]["weight"] = float(wt)
    adj = {u: [(v, R.edges[u, v]["weight"]) for v in R.successors(u)] for u in R}
    ours = dijkstra(adj, 0)
    ref = nx.single_source_dijkstra_path_length(R, 0)
    assert all(abs(ours[v] - ref[v]) < 1e-9 for v in ref) and all(ours[v] == math.inf for v in R if v not in ref)
print("OK")

## Part 4 — Failure analysis: a time conversion that changed silently (about 25 minutes)

Times such as `sched_dep_time` are stored as integers `hhmm` (e.g. 1630 for 16:30). The old notebook converted them with string
operations (audit issue S6). Run its code under the pinned pandas.

In [ ]:
# ---- the old notebook's conversion (do not edit) ----
s = flights.sched_dep_time.astype(str)
std_old = s.str.replace(r"(\d{2}$)", "") + ":" + s.str.extract(r"(\d{2}$)", expand=False) + ":00"
print(pd.DataFrame({"sched_dep_time": flights.sched_dep_time, "old conversion": std_old}).drop_duplicates().head(4).to_string(index=False))

In pandas before 2.0, `str.replace` treated the pattern as a regular expression by default, so `"1630"` became `"16"`. Since
pandas 2.0 the default is `regex=False`: the pattern is looked for **literally**, nothing matches, and `"1630"` stays `"1630"`.
No error, no warning: just wrong times. Integer arithmetic avoids strings altogether.

In [ ]:
# Task T4
# hhmm_to_minutes(x): x is a pandas Series of integer clock times hhmm (may contain missing values).
# Return minutes after midnight (0-1439) as a Series of the same length; keep missing values missing.
# 2400 (a departure at midnight at the end of the day, which occurs in dep_time) must become 0.
def hhmm_to_minutes(x):
    # YOUR CODE HERE
    raise NotImplementedError()


print(hhmm_to_minutes(pd.Series([517, 1630, 2359, 2400, None])).tolist())

In [ ]:
# Self-check for task T4: run it; it should print OK.
r = hhmm_to_minutes(pd.Series([517, 1630, 2359, 2400, None]))
assert r.iloc[:4].tolist() == [317, 990, 1439, 0] and pd.isna(r.iloc[4])
m = hhmm_to_minutes(flights.dep_time)
assert len(m) == len(flights) and m.isna().sum() == flights.dep_time.isna().sum()
assert m.dropna().between(0, 1439).all()
print("OK")

**Q2.** **Failure analysis.** Explain the two failures in Parts 2 and 4 with your numbers: what the one-line undirected construction
does to direction, repeated flights and attributes, and why the old time conversion now gives `1630:30:00`. For each, say how you
would have caught it.

*Your answer:* 

## AI-use log (required; see the course AI policy)

| Tool | What you asked or used it for | What you accepted or changed | How you verified it |
|---|---|---|---|
| *e.g. Colab AI* | *how to aggregate rows per origin and destination* | *used groupby + agg* | *T1 self-check: edge count and flight totals* |
| | | | |

If you used no AI tool, write "None".

**Cite the data** when you use them: Wickham, H. nycflights13: Flights that Departed NYC in 2013 [R package, data]. https://github.com/tidyverse/nycflights13 (CC0); originally from the US Bureau of Transportation Statistics.